# Understanding Autograd in PyTorch

This notebook explores the automatic differentiation capabilities of PyTorch, known as Autograd. Autograd is a crucial component for training neural networks, as it automatically computes gradients for backpropagation.

We will cover:
1.  **Manual Gradient Calculation**: A brief review of how gradients are calculated manually.
2.  **PyTorch Autograd Basics**: Introduction to `torch.tensor` and `requires_grad`.
3.  **Chained Rule Example**: Demonstrating autograd with a more complex function.
4.  **Manual vs. Autograd for Logistic Regression Loss**: A practical comparison.
5.  **Gradients with Vector Inputs**: How autograd handles multi-dimensional inputs.
6.  **Clearing Gradients**: Why and how to reset gradients.
7.  **Disabling Gradient Tracking**: Different methods to stop gradient computation for specific operations.

## 1. Manual Gradient Calculation: A Foundation

Before diving into PyTorch's automatic differentiation, let's briefly recall how we calculate derivatives manually. This helps in understanding what autograd does behind the scenes.

Here, we define a simple function $y = x^2$ and calculate its derivative $\frac{dy}{dx} = 2x$.

In [1]:
def dy_dx(x):
  return 2*x

In [2]:
dy_dx(3)

6

## 2.  **PyTorch Autograd Basics**: Introduction to `torch.tensor` and `requires_grad`.

#### 4 steps:
- Step-1: Intialize the x.
- Step-2: Calculate the value of Y by defining the function.
- Step-3: use Y.backward() function
- Step-4: use x.grade

In [ ]:
import torch

In [ ]:
# Step-1
x = torch.tensor(3.0, requires_grad=True)

In [ ]:
# Step-2
y = x**2

In [ ]:
x

In [ ]:
y

### The `grad_fn` Attribute

When `requires_grad` is `True`, an operation involving a tensor will create a `grad_fn` attribute on the resulting tensor. This `grad_fn` points to the function that created the tensor and knows how to compute its gradients. It's an essential part of the computational graph that Autograd builds.

-   `x` is a leaf tensor (created directly by the user), so it doesn't have a `grad_fn`.
-   `y` is a result of an operation (`x**2`), so it has a `grad_fn` (`<PowBackward0>`), indicating it was created by a power operation.

In [ ]:
# Step-3:
y.backward()

### Computing Gradients with `.backward()`

Once you have a scalar output (like a loss function) that is the result of a chain of operations, you can call `.backward()` on it. This initiates the backpropagation process, computing all gradients throughout the computational graph. The gradients are accumulated in the `.grad` attribute of all leaf tensors that had `requires_grad=True`.

In our case, `y.backward()` computes $\frac{dy}{dx}$ and stores it in `x.grad`.

In [ ]:
# Step-4:
x.grad

## 3. Chained Rule Example with Autograd


Next, let's consider a slightly more complex function $z = \sin(x^2)$ and manually calculate its derivative $\frac{dz}{dx}$.

Using the chain rule, if $y = x^2$ and $z = \sin(y)$, then:

1.  $\frac{dy}{dx} = 2x$
2.  $\frac{dz}{dy} = \cos(y) = \cos(x^2)$

So, $\frac{dz}{dx} = \frac{dz}{dy} \cdot \frac{dy}{dx} = \cos(x^2) \cdot 2x = 2x \cos(x^2)$.

The output `x.grad = tensor(6.)` is consistent with our manual calculation: $\frac{dy}{dx} = 2x$, and for $x=3$, $\frac{dy}{dx} = 2 \times 3 = 6$.

In [ ]:
import math

def dz_dx(x):
    return 2 * x * math.cos(x**2)

In [ ]:
dz_dx(4)

## 2. PyTorch Autograd Basics

PyTorch's `autograd` package provides automatic differentiation for all operations on Tensors. It's a define-by-run framework, meaning your backprop graph is built on the fly as your code runs.

To enable autograd for a tensor, set `requires_grad=True` when defining it. This tells PyTorch to track all operations involving this tensor so that gradients can be computed later.

Here, we define `x` as a PyTorch tensor and then perform an operation `y = x**2`.

In [ ]:
x = torch.tensor(4.0, requires_grad=True)

In [ ]:
y = x ** 2

In [ ]:
z = torch.sin(y)

In [ ]:
x

In [ ]:
y

In [ ]:
z

### Understanding `y.grad` Warning

When `z.backward()` is called, PyTorch calculates gradients for all tensors involved. However, you might notice that `y.grad` is `None` or produces a warning. This is because, by default, PyTorch only populates the `.grad` attribute for *leaf* tensors (tensors created by the user, like our initial `x`). Intermediate tensors in the graph, like `y`, do not have their `.grad` attributes populated unless explicitly requested using `y.retain_grad()`.

The output `x.grad = tensor(-7.6613)` matches our manual calculation for $x=4$: $\frac{dz}{dx} = 2x \cos(x^2) = 2 \times 4 \times \cos(4^2) = 8 \times \cos(16) \approx 8 \times -0.9576 = -7.6613$. This demonstrates the power of autograd in handling complex chain rules automatically.

In [ ]:
z.backward()

In [ ]:
x.grad

In [ ]:
y.grad

## 4. Manual vs. Autograd for Logistic Regression Loss

Let's apply these concepts to a common machine learning scenario: calculating gradients for a simple logistic regression model's binary cross-entropy loss. We'll first calculate the gradients manually and then compare them with PyTorch's autograd.

### Model Definition:
-   Input feature: $x$
-   True label: $y_{true}$
-   Weight: $w$
-   Bias: $b$

### Forward Pass:
1.  Linear combination: $z = w \cdot x + b$
2.  Activation (Sigmoid): $y_{pred} = \sigma(z) = \frac{1}{1 + e^{-z}}$

### Loss Function (Binary Cross-Entropy):
$L = - [y_{true} \log(y_{pred}) + (1 - y_{true}) \log(1 - y_{pred})]$

In [ ]:
import torch

# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

### Binary Cross-Entropy Loss Function

In [ ]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

### Forward Pass Calculation

In [ ]:
# Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [ ]:
loss

### Manual Gradient Calculation

Now, let's calculate the gradients of the loss with respect to the weight ($w$) and bias ($b$) using the chain rule.

1.  $\frac{\partial L}{\partial y_{pred}} = -\left( \frac{y_{true}}{y_{pred}} - \frac{1 - y_{true}}{1 - y_{pred}} \right) = \frac{y_{pred} - y_{true}}{y_{pred}(1 - y_{pred})}$
2.  $\frac{\partial y_{pred}}{\partial z} = y_{pred}(1 - y_{pred})$ (derivative of sigmoid)
3.  $\frac{\partial z}{\partial w} = x$
4.  $\frac{\partial z}{\partial b} = 1$

Combining these for the gradients of the loss:
-   $\frac{\partial L}{\partial w} = \frac{\partial L}{\partial y_{pred}} \cdot \frac{\partial y_{pred}}{\partial z} \cdot \frac{\partial z}{\partial w} = \left( \frac{y_{pred} - y_{true}}{y_{pred}(1 - y_{pred})} \right) \cdot (y_{pred}(1 - y_{pred})) \cdot x = (y_{pred} - y_{true}) \cdot x$
-   $\frac{\partial L}{\partial b} = \frac{\partial L}{\partial y_{pred}} \cdot \frac{\partial y_{pred}}{\partial z} \cdot \frac{\partial z}{\partial b} = \left( \frac{y_{pred} - y_{true}}{y_{pred}(1 - y_{pred})} \right) \cdot (y_{pred}(1 - y_{pred})) \cdot 1 = (y_{pred} - y_{true})$

In [ ]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [ ]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

### Autograd Gradient Calculation

Now, let's use PyTorch's autograd to calculate the same gradients. We need to define `w` and `b` as tensors with `requires_grad=True`.

We start by re-initializing `x` and `y` (though their `requires_grad` status doesn't directly impact the gradients of `w` and `b`).

In [ ]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

Now, we define `w` and `b` with `requires_grad=True`.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [ ]:
w

In [ ]:
b

Perform the forward pass to compute `z`, `y_pred`, and `loss`.

In [ ]:
z = w*x + b
z

In [ ]:
y_pred = torch.sigmoid(z)
y_pred

In [ ]:
loss = binary_cross_entropy_loss(y_pred, y)
loss

Call `loss.backward()` to compute the gradients automatically.

In [ ]:
loss.backward()

As you can see, the gradients computed by PyTorch's autograd (`w.grad`, `b.grad`) match the manual calculations, demonstrating its accuracy and convenience.

In [ ]:
print(w.grad)
print(b.grad)

## 5. Gradients with Vector Inputs

Autograd also seamlessly handles operations with vector or matrix inputs. When the output is a scalar (e.g., a loss function) and the input is a vector, `.backward()` computes the gradient for each element of the input vector.

Let's consider `x` as a vector and `y` as the mean of the squares of `x`'s elements: $y = \frac{1}{N} \sum_{i=1}^{N} x_i^2$.

The derivative for each $x_i$ would be $\frac{\partial y}{\partial x_i} = \frac{1}{N} \cdot 2x_i = \frac{2x_i}{N}$.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

In [ ]:
x

In [ ]:
y = (x**2).mean()
y

In [ ]:
y.backward()

In [ ]:
x.grad

## 6. Clearing Gradients (`.grad.zero_()`)

An important aspect of PyTorch's autograd is that gradients are *accumulated*. This means that if you call `.backward()` multiple times without clearing the gradients, the new gradients will be added to the existing ones in `.grad`.

This is often desirable when accumulating gradients from mini-batches, but for typical training loops, you need to explicitly clear gradients before each backward pass to prevent them from stacking up. You can do this using the `.zero_()` method on the `.grad` attribute of the tensor.

Let's see an example.

For `x = [1.0, 2.0, 3.0]` and $N=3$:
-   $\frac{\partial y}{\partial x_1} = \frac{2 \times 1}{3} = 0.6667$
-   $\frac{\partial y}{\partial x_2} = \frac{2 \times 2}{3} = 1.3333$
-   $\frac{\partial y}{\partial x_3} = \frac{2 \times 3}{3} = 2.0000$

This matches the output of `x.grad`.

In [ ]:
# clearing grad
x = torch.tensor(2.0, requires_grad=True)
x

In [ ]:
y = x ** 2
y

In [ ]:
y.backward()

In [ ]:
x.grad

Here, `x.grad` is `tensor(4.)` after the first backward pass for $y = x^2$ where $x=2$. If we were to call `y.backward()` again without clearing `x.grad`, the gradients would sum up. So, we use `x.grad.zero_()` to reset it.

In [ ]:
x.grad.zero_()

After `x.grad.zero_()`, `x.grad` is reset to `tensor(0.)`. This ensures that subsequent gradient computations start fresh.

In [ ]:
# disable gradient tracking
x = torch.tensor(2.0, requires_grad=True)
x

## 7. Disabling Gradient Tracking

There are situations where you might want to temporarily disable gradient tracking. This is common during inference (when you don't need to compute gradients) or when performing operations that should not be part of the gradient computation (e.g., updating model parameters). Disabling gradient tracking can save memory and computation time.

PyTorch provides several ways to do this:

1.  **`tensor.requires_grad_(False)`**: Changes the `requires_grad` attribute of a tensor in-place.
2.  **`tensor.detach()`**: Creates a new tensor that shares the same data as the original but does not require gradients and is detached from the computational graph.
3.  **`torch.no_grad()`**: A context manager that temporarily disables gradient tracking for all operations within its block. (This was discussed in detail in the previous output).

In [ ]:
y = x ** 2
y

In [ ]:
y.backward()

In [ ]:
x.grad

In [ ]:
# option 1 - requires_grad_(False)
# option 2 - detach()
# option 3 - torch.no_grad()

### Option 1: `tensor.requires_grad_(False)`

This method changes the `requires_grad` attribute of a tensor in-place. Once set to `False`, operations involving this tensor will no longer track gradients.

Let's apply this to `x`.

## Disabling Gradient Tracking: `torch.no_grad()`

Sometimes, you might want to perform operations without tracking gradients. This is particularly useful during inference when you don't need to compute derivatives, or when you're updating model parameters and don't want the update operation itself to be part of the gradient computation.

`torch.no_grad()` is a context manager that temporarily sets all of its contained operations to have `requires_grad=False`. This means that any tensor created or modified within this block will not have gradient tracking enabled.

### Mathematical Principle (for operations inside `no_grad`):
If $y = f(x)$ is an operation performed within a `torch.no_grad()` block, and $x$ has `requires_grad=True`, then the computational graph for $y$ will be detached from $x$. Effectively, $\frac{dy}{dx}$ will not be computed or tracked.

This saves memory and computation, as intermediate activations that would normally be stored for backward pass are not retained.

In [ ]:
x = torch.tensor(5.0, requires_grad=True)
x_nograd_option = x
print(f"Initial x: {x_nograd_option}")

with torch.no_grad():
    y_nograd = x_nograd_option ** 2

print(f"y (inside no_grad): {y_nograd}")

# Try to perform backward pass on y_nograd
try:
    y_nograd.backward()
except RuntimeError as e:
    print(f"\nCaught an expected error when calling backward on y_nograd: {e}")

print(f"x.grad after y_nograd.backward(): {x_nograd_option.grad}")

# Compare with an operation outside no_grad
y_grad = x ** 3
y_grad.backward()

print(f"y (outside no_grad): {y_grad}")
print(f"x.grad after y_grad.backward(): {x_nograd_option.grad}")

In [ ]:
x.requires_grad_(False)

Now that `x.requires_grad` is `False`, any operation deriving from `x` will also not have gradients tracked. Trying to call `.backward()` on such a tensor will result in a `RuntimeError`.

In [ ]:
x

In [ ]:
y = x ** 2

In [ ]:
y

As expected, trying to call `y.backward()` results in a `RuntimeError` because `y` no longer has a `grad_fn` and doesn't require gradients, as its input `x` explicitly had gradient tracking disabled.

In [ ]:
y.backward()

### Option 2: `tensor.detach()`

The `.detach()` method returns a new tensor, detached from the current computational graph. The new tensor will have `requires_grad=False`, even if the original tensor had `requires_grad=True`. The data of the new tensor will share the same memory with the original tensor.

Let's see how `detach()` works. We'll start with a tensor `x` that `requires_grad`.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
x

Here, `z` is a detached version of `x`. Notice that `z` does not have `requires_grad=True`.

Now, let's create `y` from the original `x` and `y1` from the detached `z`.

In [ ]:
z = x.detach()
z

In [ ]:
y = x ** 2

In [ ]:
y

As you can see, `y` (derived from `x`) still has a `grad_fn` (`<PowBackward0>`), so its gradients can be computed.

However, `y1` (derived from `z`, which was detached from `x`) does not have a `grad_fn` and thus does not track gradients.

In [ ]:
y1 = z ** 2
y1

In [ ]:
y.backward()

Calling `y.backward()` works fine because `y` is part of the computational graph rooted at `x`. However, trying to call `y1.backward()` results in an error because `y1` was created from `z`, which was detached, effectively breaking the gradient tracking path from `y1` back to `x`.

In [ ]:
y1.backward()

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
x

In [ ]:
y = x ** 2

In [ ]:
y

In [ ]:
y.backward()